In [ ]:
import cv2
import os
from ultralytics import YOLO

In [4]:
pip install moviepy


Note: you may need to restart the kernel to use updated packages.


In [8]:
!pip uninstall moviepy


^C


In [7]:
from moviepy.editor import VideoFileClip, concatenate_videoclips


ModuleNotFoundError: No module named 'moviepy.editor'

In [4]:
pip install safetensors

Note: you may need to restart the kernel to use updated packages.


In [5]:
import cv2
import csv
from ultralytics import YOLO
import os 

# === Paths ===
VIDEO_PATH = r"C:\Users\Nitro v15\Desktop\volly high\input/watch.mp4"
model_paths = {
    "action": r"C:\Users\Nitro v15\Desktop\volly high\weights\action_detection.pt",
    "ball": r"C:\Users\Nitro v15\Desktop\volly high\weights\ball_segment.pt",
    "court": r"C:\Users\Nitro v15\Desktop\volly high\weights\court_segment.pt",
    "game": r"C:\Users\Nitro v15\Desktop\volly high\weights\game_state.safetensors"
}


# Output CSV files
DETECTION_LOG_CSV = 'output/detection_log.csv'
GAME_STATE_LOG_CSV = 'output/game_state_log.csv'

# === Load models ===
models = {key: YOLO(path) for key, path in model_paths.items()}
print("Models loaded:")
for k, m in models.items():
    print(f" - {k}: {m.names}")

# === Open video ===
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_num = 0

# === State tracking ===
cumulative_counts = {
    key: {name: 0 for name in model.names.values()}
    for key, model in models.items()
}
last_game_state = None
game_state_changes = []

# Prepare CSV files
with open(DETECTION_LOG_CSV, 'w', newline='') as det_file, \
     open(GAME_STATE_LOG_CSV, 'w', newline='') as gs_file:

    det_writer = csv.writer(det_file)
    gs_writer = csv.writer(gs_file)

    # Write CSV headers
    det_writer.writerow([
        'frame', 'time_sec', 'model', 'class', 'confidence',
        'x1', 'y1', 'x2', 'y2'
    ])
    gs_writer.writerow(['frame', 'time_sec', 'game_state'])

    print("\nStarting video analysis with logging...\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_num += 1
        time_sec = frame_num / fps

        # Run inference for each model on the current frame
        for key, model in models.items():
            results = model.predict(source=frame, conf=0.5)
            if results and results[0].boxes:
                for box in results[0].boxes:
                    cls_id = int(box.cls[0])
                    conf = float(box.conf[0])
                    label = model.names[cls_id]
                    x1, y1, x2, y2 = box.xyxy[0].tolist()

                    # Update counts
                    cumulative_counts[key][label] += 1

                    # Log detection to CSV
                    det_writer.writerow([
                        frame_num, f"{time_sec:.3f}", key, label, f"{conf:.3f}",
                        f"{x1:.1f}", f"{y1:.1f}", f"{x2:.1f}", f"{y2:.1f}"
                    ])

                    # For game_state model, track state changes
                    if key == 'game':
                        if label != last_game_state:
                            last_game_state = label
                            game_state_changes.append((frame_num, time_sec, label))
                            gs_writer.writerow([frame_num, f"{time_sec:.3f}", label])

        # Optional: print progress every 100 frames
        if frame_num % 100 == 0:
            print(f"Processed {frame_num} frames...")

    print("\nVideo analysis complete.")

cap.release()

# === Print cumulative counts summary ===
print("\n=== Detection Counts Summary ===")
for key, counts in cumulative_counts.items():
    print(f"\nModel: {key}")
    for label, count in counts.items():
        if count > 0:
            print(f"  {label}: {count}")

# === Print game state changes summary ===
print("\n=== Game State Changes ===")
for fnum, tsec, state in game_state_changes:
    print(f"  Frame {fnum} / {tsec:.2f}s : {state}")


WARNING Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Models loaded:
 - action: {0: 'ball', 1: 'block', 2: 'receive', 3: 'set', 4: 'spike', 5: 'serve'}
 - ball: {0: 'ball'}
 - court: {0: 'Court'}


TypeError: model='C:\Users\Nitro v15\Desktop\volly high\weights\game_state.safetensors' is not a supported model format. Ultralytics supports: ('PyTorch', 'TorchScript', 'ONNX', 'OpenVINO', 'TensorRT', 'CoreML', 'TensorFlow SavedModel', 'TensorFlow GraphDef', 'TensorFlow Lite', 'TensorFlow Edge TPU', 'TensorFlow.js', 'PaddlePaddle', 'MNN', 'NCNN', 'IMX', 'RKNN')
See https://docs.ultralytics.com/modes/predict for help.

In [1]:
import cv2
import csv
from ultralytics import YOLO
import os

# === Paths ===
VIDEO_PATH = r"C:\Users\Nitro v15\Desktop\volly high\input/watch.mp4"
OUTPUT_VIDEO_PATH = 'output/output.mp4'
model_paths = {
    "action": r"C:\Users\Nitro v15\Desktop\volly high\weights\action_detection.pt",
    "ball": r"C:\Users\Nitro v15\Desktop\volly high\weights\ball_segment.pt"
}

DETECTION_LOG_CSV = 'output/detection_log.csv'
os.makedirs('output', exist_ok=True)

# === Load models ===
models = {key: YOLO(path) for key, path in model_paths.items()}
print("Models loaded:")
for k, m in models.items():
    print(f" - {k}: {m.names}")

# === Open video ===
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
frame_num = 0

# CSV writer
with open(DETECTION_LOG_CSV, 'w', newline='') as det_file:
    det_writer = csv.writer(det_file)
    det_writer.writerow([
        'frame', 'time_sec', 'model', 'class', 'confidence',
        'x1', 'y1', 'x2', 'y2'
    ])

    print("\nStarting video processing...\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_num += 1
        time_sec = frame_num / fps

        # Placeholder values to display on screen
        current_action = "None"
        current_ball_pos = "Not Visible"

        for key, model in models.items():
            results = model.predict(source=frame, conf=0.5, verbose=False)
            if results and results[0].boxes:
                for box in results[0].boxes:
                    cls_id = int(box.cls[0])
                    conf = float(box.conf[0])
                    label = model.names[cls_id]
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

                    # Write to CSV
                    det_writer.writerow([
                        frame_num, f"{time_sec:.2f}", key, label, f"{conf:.3f}",
                        x1, y1, x2, y2
                    ])

                    # Update action or ball status
                    if key == "action" and conf > 0.6:
                        current_action = f"{label} ({conf:.2f})"
                        # Draw bounding box for action (optional)
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 165, 0), 2)
                        cv2.putText(frame, label, (x1, y1 - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 165, 0), 2)

                    if key == "ball":
                        current_ball_pos = f"x:{(x1 + x2)//2}, y:{(y1 + y2)//2}"
                        # Draw ball position
                        cv2.circle(frame, ((x1 + x2)//2, (y1 + y2)//2), 8, (0, 255, 0), -1)

        # === Add overlay text ===
        cv2.putText(frame, f"Action: {current_action}", (20, 40),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 0, 255), 2)
        cv2.putText(frame, f"Ball Pos: {current_ball_pos}", (20, 80),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 255, 255), 2)
        cv2.putText(frame, "State: Ongoing", (20, 120),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (100, 100, 255), 2)

        out_writer.write(frame)

        if frame_num % 100 == 0:
            print(f"Processed {frame_num} frames...")

    print("\n✅ Done! Video saved to:", OUTPUT_VIDEO_PATH)

cap.release()
out_writer.release()


Models loaded:
 - action: {0: 'ball', 1: 'block', 2: 'receive', 3: 'set', 4: 'spike', 5: 'serve'}
 - ball: {0: 'ball'}

Starting video processing...

Processed 100 frames...
Processed 200 frames...
Processed 300 frames...
Processed 400 frames...
Processed 500 frames...
Processed 600 frames...
Processed 700 frames...
Processed 800 frames...
Processed 900 frames...
Processed 1000 frames...
Processed 1100 frames...
Processed 1200 frames...
Processed 1300 frames...

✅ Done! Video saved to: output/output.mp4


In [ ]:
import cv2
import csv
import os
from ultralytics import YOLO

# === Paths ===
VIDEO_PATH = r"C:\Users\Nitro v15\Desktop\volly high\input\watch.mp4"
OUTPUT_VIDEO_PATH = 'output/output.mp4'
DETECTION_LOG_CSV = 'output/detection_log.csv'

model_paths = {
    "action": r"C:\Users\Nitro v15\Desktop\volly high\weights\action_detection.pt",
    "ball": r"C:\Users\Nitro v15\Desktop\volly high\weights\ball_segment.pt"
}

# === Create output folder ===
os.makedirs('output', exist_ok=True)

# === Load YOLO models ===
models = {name: YOLO(path) for name, path in model_paths.items()}
print("✅ Models loaded:")
for name, model in models.items():
    print(f" - {name}: {model.names}")

# === Open video ===
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
frame_num = 0

# === CSV logging ===
with open(DETECTION_LOG_CSV, 'w', newline='') as det_file:
    det_writer = csv.writer(det_file)
    det_writer.writerow(['frame', 'time_sec', 'model', 'class', 'confidence', 'x1', 'y1', 'x2', 'y2'])

    print("\n🎥 Starting video processing...\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_num += 1
        time_sec = frame_num / fps

        current_action = "None"
        current_ball_pos = "Not visible"

        for key, model in models.items():
            results = model.predict(frame, conf=0.5, verbose=False)
            boxes = results[0].boxes

            if boxes is not None:
                for box in boxes:
                    cls_id = int(box.cls[0])
                    label = model.names[cls_id]
                    conf = float(box.conf[0])
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

                    # Log detection
                    det_writer.writerow([
                        frame_num, f"{time_sec:.2f}", key, label, f"{conf:.3f}", x1, y1, x2, y2
                    ])

                    # Annotate frame
                    if key == "action" and conf > 0.6:
                        current_action = f"{label} ({conf:.2f})"
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 165, 0), 2)
                        cv2.putText(frame, label, (x1, y1 - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 165, 0), 2)

                    if key == "ball":
                        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                        current_ball_pos = f"x:{cx}, y:{cy}"
                        cv2.circle(frame, (cx, cy), 8, (0, 255, 0), -1)

        # === Add overlay text ===
        cv2.putText(frame, f"Action: {current_action}", (20, 40),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 0, 255), 2)
        cv2.putText(frame, f"Ball Pos: {current_ball_pos}", (20, 80),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 255, 255), 2)
        cv2.putText(frame, "State: Ongoing", (20, 120),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (100, 100, 255), 2)

        out_writer.write(frame)

        if frame_num % 100 == 0:
            print(f"Processed {frame_num} frames...")

    print("\n✅ Done! Output saved to:", OUTPUT_VIDEO_PATH)

cap.release()
out_writer.release()
import cv2
import csv
import os
from ultralytics import YOLO

# === Paths ===
VIDEO_PATH = r"C:\Users\Nitro v15\Desktop\volly high\input\watch.mp4"
OUTPUT_VIDEO_PATH = 'output/output.mp4'
DETECTION_LOG_CSV = 'output/detection_log.csv'

model_paths = {
    "action": r"C:\Users\Nitro v15\Desktop\volly high\weights\action_detection.pt",
    "ball": r"C:\Users\Nitro v15\Desktop\volly high\weights\ball_segment.pt"
}

# === Create output folder ===
os.makedirs('output', exist_ok=True)

# === Load YOLO models ===
models = {name: YOLO(path) for name, path in model_paths.items()}
print("✅ Models loaded:")
for name, model in models.items():
    print(f" - {name}: {model.names}")

# === Open video ===
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out_writer = cv2.VideoWriter(OUTPUT_VIDEO_PATH, fourcc, fps, (frame_width, frame_height))
frame_num = 0

# === CSV logging ===
with open(DETECTION_LOG_CSV, 'w', newline='') as det_file:
    det_writer = csv.writer(det_file)
    det_writer.writerow(['frame', 'time_sec', 'model', 'class', 'confidence', 'x1', 'y1', 'x2', 'y2'])

    print("\n🎥 Starting video processing...\n")

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_num += 1
        time_sec = frame_num / fps

        current_action = "None"
        current_ball_pos = "Not visible"

        for key, model in models.items():
            results = model.predict(frame, conf=0.5, verbose=False)
            boxes = results[0].boxes

            if boxes is not None:
                for box in boxes:
                    cls_id = int(box.cls[0])
                    label = model.names[cls_id]
                    conf = float(box.conf[0])
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

                    # Log detection
                    det_writer.writerow([
                        frame_num, f"{time_sec:.2f}", key, label, f"{conf:.3f}", x1, y1, x2, y2
                    ])

                    # Annotate frame
                    if key == "action" and conf > 0.6:
                        current_action = f"{label} ({conf:.2f})"
                        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 165, 0), 2)
                        cv2.putText(frame, label, (x1, y1 - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 165, 0), 2)

                    if key == "ball":
                        cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                        current_ball_pos = f"x:{cx}, y:{cy}"
                        cv2.circle(frame, (cx, cy), 8, (0, 255, 0), -1)

        # === Add overlay text ===
        cv2.putText(frame, f"Action: {current_action}", (20, 40),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 0, 255), 2)
        cv2.putText(frame, f"Ball Pos: {current_ball_pos}", (20, 80),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 255, 255), 2)
        cv2.putText(frame, "State: Ongoing", (20, 120),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, (100, 100, 255), 2)

        out_writer.write(frame)

        if frame_num % 100 == 0:
            print(f"Processed {frame_num} frames...")

    print("\n✅ Done! Output saved to:", OUTPUT_VIDEO_PATH)

cap.release()
out_writer.release()


✅ Models loaded:
 - action: {0: 'ball', 1: 'block', 2: 'receive', 3: 'set', 4: 'spike', 5: 'serve'}
 - ball: {0: 'ball'}

🎥 Starting video processing...

Processed 100 frames...
Processed 200 frames...
Processed 300 frames...
Processed 400 frames...
Processed 500 frames...
Processed 600 frames...
Processed 700 frames...
Processed 800 frames...
Processed 900 frames...
Processed 1000 frames...
